#### Requirements

In [1]:
# !conda install jupyter
# !conda install pandas
# !pip install python-dotenv
# !pip install langchain
# !pip install langchain-community
# !pip install langchain-openai
# !pip install langchain-text-splitters
# !pip install langchain-ollama
# !pip install pydantic
# !pip install chromadb
# !pip install pypdf

# !ollama run llama3.2   
# !ollama pull llama3.2:3b
# !ollama pull mxbai-embed-large
# !ollama list

#### Preparation stage

In [2]:
# =========================================================
# 1) PREPARATION STAGE
# =========================================================
import os
import json
import re
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Dict, Any

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

# New Chroma client API
from chromadb import PersistentClient

# -------------------
# Environment & models
# -------------------
load_dotenv()

EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "mxbai-embed-large")
CHAT_MODEL = os.environ.get("CHAT_MODEL", "llama3.2:3b")

embedding_function = OllamaEmbeddings(model=EMBEDDING_MODEL)

# Force strict JSON output from the model for reliable parsing
llm = ChatOllama(
    model=CHAT_MODEL,
    temperature=0,
    top_p=1,
    num_ctx=4096,
    format="json",     # strict JSON responses
    seed=42,           # optional determinism
)

# -------------------
# Vectorstore housekeeping (fresh, writable path per run)
# -------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
GLOBAL_DB_PATH = Path.cwd() / f"vectorstore_chroma_{timestamp}"
GLOBAL_DB_PATH.mkdir(parents=True, exist_ok=True)
if not os.access(str(GLOBAL_DB_PATH), os.W_OK):
    raise PermissionError(f"Persist directory is not writable: {GLOBAL_DB_PATH}")

# -------------------
# Load & chunk PDFs
# -------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", " ", ""]
)

pdf_dir = Path("./data/exampleFiles")
if not pdf_dir.exists():
    raise FileNotFoundError(f"PDF directory not found: {pdf_dir}")

all_chunks = []
chunks_by_doc = defaultdict(list)

for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        path = pdf_dir / filename
        pages = PyPDFLoader(str(path)).load()  # per-page docs w/ metadata (source, page)
        chunks = text_splitter.split_documents(pages)

        # Add provenance metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata["source"] = filename
            # page usually present via PyPDFLoader; keep chunk_id for provenance
            chunk.metadata["chunk_id"] = i

        chunks_by_doc[filename] = chunks
        all_chunks.extend(chunks)

print(f"Loaded total chunks: {len(all_chunks)}")

# -------------------
# Build Chroma vectorstore with PersistentClient
# -------------------
client = PersistentClient(path=str(GLOBAL_DB_PATH))
collection_name = f"slr_{timestamp}"

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_function,
    client=client,
    collection_name=collection_name,
)
print("Vectorstore ready with:", len(vectorstore.get()["documents"]), "documents")

# =========================================================
# 2) SCHEMA, PROMPT, RETRIEVAL & GUARDS
# =========================================================

class SupportingSpan(BaseModel):
    source: str = Field(..., description="Filename of the PDF")
    page: Optional[int] = Field(None, description="0-based page number if available")
    chunk_id: Optional[int] = Field(None, description="Chunk ID if available")
    quote: str = Field(..., description="Exact quoted text")

class AnswerWithSources(BaseModel):
    decision: Optional[int] = Field(None, description="1=include, 0=exclude, null if manual review")
    reasoning: str = ""
    sources: Optional[str] = ""
    supporting_spans: List[SupportingSpan] = Field(default_factory=list)
    confidence: Optional[float] = Field(None, ge=0.0, le=1.0)
    requires_manual_review: bool = False
    prompt_version: str = "v1.5"
    model: str = ""

# ---------- Prompt (escape literal braces in examples) ----------
PROMPT_TEMPLATE = """
You are a careful researcher deciding whether to include a paper in a systematic literature review
about machine learning applications in the port call process.

STRICT RULES:
- Only use information that appears in the provided CONTEXT.
- If the context does not contain enough evidence to support a decision,
  set "requires_manual_review": true, "decision": null, and explain why.
- Cite exact quotes from the context as supporting_spans and provide their provenance (source, page, chunk_id).
- Always set "sources" to the filename(s) visible in the CONTEXT headers; if unsure, use the current document's filename.
- Be concise and avoid speculation.
- Use double quotes around ALL string values. Do NOT include trailing commas.

INCLUSION POLICY:
- If evidence in the provided CONTEXT supports applicability to port call process optimization
  and your confidence is at least 0.8, prefer "decision": 1 (include).
- If evidence is partial or indirect, you may set "requires_manual_review": true with a lower confidence.

IMPORTANT: Copy quotes exactly as they appear in CONTEXT. Do not paraphrase. If you cannot find an exact quote, leave the quote field empty.

Return ONLY valid JSON with these keys:
{{
  "decision": 1 or 0 or null,
  "reasoning": "...",
  "sources": "filename(s) for this context",
  "supporting_spans": [
     {{"source":"...", "page": 0, "chunk_id": 12, "quote":"exact text from context"}}
  ],
  "confidence": 0.0-1.0,
  "requires_manual_review": true/false
}}

--- CONTEXT (top retrieved chunks for this document) ---
{context}
--- END CONTEXT ---

Question: {question}

When you can make an explicit decision, set "decision": 1 (include) or 0 (exclude),
and provide a brief reasoning (2–4 sentences) and a confidence in [0,1].
"""
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

# ---------- Helpers ----------
def format_docs(docs):
    lines = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        chunk_id = d.metadata.get("chunk_id", None)
        header = f"[SOURCE={src}, PAGE={page}, CHUNK={chunk_id}]"
        lines.append(f"{header}\n{d.page_content}")
    return "\n\n---\n\n".join(lines)

def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

def verify_quotes_in_context(quotes: List[str], docs: List):
    context_blob = "\n\n".join(d.page_content for d in docs)
    norm_ctx = normalize_text(context_blob)
    missing = []
    for q in quotes:
        if normalize_text(q) not in norm_ctx:
            missing.append(q)
    return missing

# Retrieval parameters — calibrate threshold to your backend’s score semantics.
K = 12
SCORE_THRESHOLD = 0.2
SCORE_IS_DISTANCE = False  # set True if backend returns distance (lower is better)

def retrieve_for_document(vectorstore, query: str, doc_name: str, k: int = K,
                          threshold: float = SCORE_THRESHOLD, score_is_distance: bool = SCORE_IS_DISTANCE):
    results = vectorstore.similarity_search_with_score(query, k=k, filter={"source": doc_name})
    if score_is_distance:
        filtered = [(doc, score) for doc, score in results if float(score) <= threshold]
    else:
        filtered = [(doc, score) for doc, score in results if float(score) >= threshold]
    docs = [d for d, s in filtered]
    scores = [float(s) for d, s in filtered]
    return docs, scores, {"k": k, "threshold": threshold, "filter": {"source": doc_name},
                          "search": "similarity_with_score", "score_is_distance": score_is_distance}

# =========================================================
# 3) EXECUTION: QUESTIONS, DECISIONS, LOGGING
# =========================================================

questions = [
    "Should the paper be included in a systematic literature review about machine learning applications in a port call process?",
]

rows = []
evidence_per_doc = {}

for doc_name in chunks_by_doc.keys():
    print(f"\nProcessing: {doc_name}")
    row = {"document": doc_name}

    for i, q in enumerate(questions, 1):
        docs, scores, retrieval_params = retrieve_for_document(
            vectorstore, q, doc_name, k=K, threshold=SCORE_THRESHOLD, score_is_distance=SCORE_IS_DISTANCE
        )

        if not docs:
            # Abstain when no strong evidence retrieved
            result = AnswerWithSources(
                decision=None,
                requires_manual_review=True,
                reasoning="No sufficiently relevant chunks were retrieved for this document.",
                sources=doc_name,
                supporting_spans=[],
                confidence=None,
                prompt_version="v1.5",
                model=CHAT_MODEL,
            )
            raw_json = {"decision": None, "reasoning": result.reasoning, "sources": doc_name,
                        "supporting_spans": [], "confidence": None, "requires_manual_review": True}
        else:
            context_text = format_docs(docs)
            # With format="json", model MUST return JSON
            raw_msg = (prompt | llm).invoke({"context": context_text, "question": q})
            raw_text = getattr(raw_msg, "content", raw_msg)

            # Parse raw JSON strictly (with a small fallback)
            try:
                raw_json = json.loads(raw_text)
            except Exception:
                # fallback: try to extract object region
                start = raw_text.find("{")
                end = raw_text.rfind("}")
                try:
                    raw_json = json.loads(raw_text[start:end+1])
                except Exception as e:
                    raw_json = {
                        "decision": None,
                        "reasoning": f"Unexpected non-JSON output: {e}",
                        "sources": doc_name,
                        "supporting_spans": [],
                        "confidence": None,
                        "requires_manual_review": True
                    }

            # Backfill & validate via Pydantic
            def backfill_missing_fields(d: Dict[str, Any], doc_name: str) -> Dict[str, Any]:
                d = dict(d)
                d.setdefault("decision", None)
                d.setdefault("reasoning", "")
                d.setdefault("supporting_spans", [])
                d.setdefault("confidence", None)
                d.setdefault("requires_manual_review", False)
                if not d.get("sources"):
                    uniq_sources = {
                        s.get("source")
                        for s in d.get("supporting_spans", [])
                        if isinstance(s, dict) and s.get("source")
                    }
                    d["sources"] = ", ".join(sorted(uniq_sources)) if uniq_sources else doc_name
                # normalize spans structure
                spans = []
                for s in d.get("supporting_spans", []):
                    if isinstance(s, dict):
                        spans.append({
                            "source": s.get("source", doc_name),
                            "page": s.get("page"),
                            "chunk_id": s.get("chunk_id"),
                            "quote": s.get("quote", ""),
                        })
                    else:
                        spans.append({"source": doc_name, "page": None, "chunk_id": None, "quote": str(s)})
                d["supporting_spans"] = spans
                if d["decision"] is None and d["requires_manual_review"] is False:
                    d["requires_manual_review"] = True
                return d

            raw_json = backfill_missing_fields(raw_json, doc_name)
            result = AnswerWithSources(**raw_json)
            result.model = CHAT_MODEL

            # Verify quotes are present; if not, flip to manual review
            missing = verify_quotes_in_context([s["quote"] for s in raw_json.get("supporting_spans", [])], docs)
            if missing and not result.requires_manual_review:
                result.requires_manual_review = True
                result.decision = None
                result.reasoning += f" (Flagged for manual review: {len(missing)} quoted span(s) not found in retrieved context.)"

        # Populate summary row
        row[f"Q{i}_Decision"] = result.decision
        row[f"Q{i}_RequiresManualReview"] = result.requires_manual_review
        row[f"Q{i}_Confidence"] = result.confidence
        row[f"Q{i}_Reasoning"] = result.reasoning
        row[f"Q{i}_Sources"] = result.sources

        # Persist rich evidence (store raw JSON and validated object)
        evidence_per_doc.setdefault(doc_name, []).append({
            "question": q,
            "raw_json": raw_json,  # strict JSON the model produced (or fallback)
            "supporting_spans": raw_json.get("supporting_spans", []),
            "retrieved": [
                {
                    "source": d.metadata.get("source"),
                    "page": d.metadata.get("page"),
                    "chunk_id": d.metadata.get("chunk_id"),
                    "score": scores[j] if j < len(scores) else None,
                    "excerpt": d.page_content[:800],
                }
                for j, d in enumerate(docs)
            ],
            "retrieval_params": retrieval_params,
            "prompt_version": result.prompt_version,
            "model": result.model,
        })

    rows.append(row)

# =========================================================
# 4) OUTPUTS: CSV (summary) + JSON (evidence) + Option 4 spans columns
# =========================================================

df = pd.DataFrame(rows).set_index("document")
print("\n--- Summary preview ---")
print(df.head(20))

# -------- Option 4: both JSON and human-readable spans columns --------
Q1 = "Should the paper be included in a systematic literature review about machine learning applications in a port call process?"

def collect_supporting_spans(evidence_per_doc: Dict[str, Any], question_text: str):
    """Return a dict: {doc_name: list_of_spans} filtered for a specific question."""
    out = {}
    for doc_name, entries in evidence_per_doc.items():
        spans = []
        for e in entries:
            if e.get("question") == question_text:
                spans.extend(e.get("supporting_spans", []))
        out[doc_name] = spans
    return out

supporting_spans_by_doc = collect_supporting_spans(evidence_per_doc, Q1)

# JSON-serialized spans (compact & machine-readable)
df["Q1_SupportingSpans"] = df.index.map(
    lambda doc: json.dumps(supporting_spans_by_doc.get(doc, []), ensure_ascii=False)
)

# Human-readable formatted spans (nice in reports)
def format_span(s: Dict[str, Any]) -> str:
    src = s.get("source", "")
    page = s.get("page", "")
    chunk_id = s.get("chunk_id", "")
    quote = (s.get("quote") or "").strip().replace("\n", " ")
    # Truncate long quotes for display (full quotes remain in JSON column)
    if len(quote) > 300:
        quote = quote[:297] + "..."
    return f'[src={src}, page={page}, chunk={chunk_id}] "{quote}"'

def spans_to_text(spans: List[Dict[str, Any]]) -> str:
    if not spans:
        return ""
    return "\n".join(format_span(s) for s in spans)

df["Q1_SupportingSpansText"] = df.index.map(
    lambda doc: spans_to_text(supporting_spans_by_doc.get(doc, []))
)

# # Save outputs
# csv_path = Path.cwd() / f"slr_screening_{timestamp}.csv"
# json_path = Path.cwd() / f"slr_screening_evidence_{timestamp}.json"
# csv_with_spans_path = Path.cwd() / f"slr_screening_w_spans_{timestamp}.csv"

# df.to_csv(csv_path, index=True)
# with open(json_path, "w", encoding="utf-8") as f:
#     json.dump(evidence_per_doc, f, ensure_ascii=False, indent=2)

# # Save the DataFrame again, now with spans columns
# df.to_csv(csv_with_spans_path, index=True)

# print(f"\nSaved summary CSV → {csv_path}")
# print(f"Saved evidence JSON → {json_path}")
# print(f"Saved CSV with supporting spans → {csv_with_spans_path}")
# print(f"Chroma persisted at → {GLOBAL_DB_PATH}")


Loaded total chunks: 290
Vectorstore ready with: 290 documents

Processing: SLR_liang_aisclean_trajPred_2024.pdf

Processing: testIrrelevanpaper1.pdf

Processing: liu_trajClustering_safetyDistance_2024_SLR.pdf

--- Summary preview ---
                                                Q1_Decision  \
document                                                      
SLR_liang_aisclean_trajPred_2024.pdf                    1.0   
testIrrelevanpaper1.pdf                                 NaN   
liu_trajClustering_safetyDistance_2024_SLR.pdf          1.0   

                                                Q1_RequiresManualReview  \
document                                                                  
SLR_liang_aisclean_trajPred_2024.pdf                              False   
testIrrelevanpaper1.pdf                                            True   
liu_trajClustering_safetyDistance_2024_SLR.pdf                    False   

                                                Q1_Confidence  \
document

In [3]:
df

,Q1_Decision,Q1_RequiresManualReview,Q1_Confidence,Q1_Reasoning,Q1_Sources,Q1_SupportingSpans,Q1_SupportingSpansText
document,,,,,,,
SLR_liang_aisclean_trajPred_2024.pdf,1.0,False,0.9,The paper discusses the application of machine...,SLR_liang_aisclean_trajPred_2024.pdf,[],
testIrrelevanpaper1.pdf,NaN,True,0.0,The provided context does not contain enough e...,testIrrelevanpaper1.pdf,[],
liu_trajClustering_safetyDistance_2024_SLR.pdf,1.0,False,0.9,The paper discusses the application of machine...,liu_trajClustering_safetyDistance_2024_SLR.pdf,"[{""source"": ""liu_trajClustering_safetyDistance...",[src=liu_trajClustering_safetyDistance_2024_SL...


In [4]:
for index, row in df.iterrows():
    print(f"Document: {index}")
    print(f"Q1_Reasoning: {row['Q1_Reasoning']}\n")

Document: SLR_liang_aisclean_trajPred_2024.pdf
Q1_Reasoning: The paper discusses the application of machine learning techniques to analyze AIS data for maritime safety and efficiency, which is relevant to the port call process. The authors also highlight the importance of addressing missing and noise data in AIS data, which is a common challenge in maritime data mining. Given the relevance of the topic and the potential value of the research, I believe that this paper should be included in a systematic literature review on machine learning applications in a port call process.

Document: testIrrelevanpaper1.pdf
Q1_Reasoning: The provided context does not contain enough evidence to support a decision about including the paper in a systematic literature review about machine learning applications in a port call process. The context is focused on long-read sequencing and its application in cancer genomics, but it does not mention machine learning or port call processes.

Document: liu_trajC